In [ ]:
# Install once if required
# %pip install tensorflow matplotlib

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.datasets import mnist

tf.random.set_seed(42)
np.random.seed(42)

# 1. Load and preprocess image data
(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train = x_train.astype("float32")[..., None] / 255.0
x_test = x_test.astype("float32")[..., None] / 255.0

y_train_c = keras.utils.to_categorical(y_train, 10)
y_test_c = keras.utils.to_categorical(y_test, 10)

print("Training data:", x_train.shape)
print("Testing data:", x_test.shape)

# 2. Define CNN architecture
model = keras.Sequential([
    layers.Input(shape=(28, 28, 1)),
    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dropout(0.5),
    layers.Dense(10, activation="softmax")
])
model.summary()

# 3. Compile and train
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)
history = model.fit(
    x_train, y_train_c,
    batch_size=128,
    epochs=10,
    validation_split=0.1,
    verbose=2
)

# 4. Evaluate performance
loss, accuracy = model.evaluate(x_test, y_test_c, verbose=0)
print("Test loss:", round(loss, 4))
print("Test accuracy:", round(accuracy, 4))

# Plot training and validation loss/accuracy
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(history.history["loss"], label="Training loss")
ax[0].plot(history.history["val_loss"], label="Validation loss")
ax[0].set_title("Model Loss")
ax[0].set_xlabel("Epoch")
ax[0].legend()

ax[1].plot(history.history["accuracy"], label="Training accuracy")
ax[1].plot(history.history["val_accuracy"], label="Validation accuracy")
ax[1].set_title("Model Accuracy")
ax[1].set_xlabel("Epoch")
ax[1].legend()
plt.tight_layout()
plt.show()

# Predict first 10 test images
predictions = model.predict(x_test[:10], verbose=0).argmax(axis=1)
print("Predicted labels:", predictions)
print("Actual labels:   ", y_test[:10])

fig, axes = plt.subplots(1, 10, figsize=(12, 2))
for i, axis in enumerate(axes):
    axis.imshow(x_test[i].squeeze(), cmap="gray")
    axis.set_title(str(predictions[i]))
    axis.axis("off")
plt.tight_layout()
plt.show()
